# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.datePublished}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema may contain multiple record sets. Let's list all available record sets and the fields (columns) available within each of them. **All elements are referenced by their `@id` as per best practices.**

In [ ]:
# List all record sets by @id with brief details

record_sets = dataset.record_sets  # This will be a dict {recordset_id: RecordSet}

print(f"Found {len(record_sets)} record set(s):\n")
for rs_id, rs in record_sets.items():
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {getattr(rs, 'description', '')}")
    # List fields in this record set
    print("  Fields/columns (@id : name : dataType):")
    for field_id, field in rs.fields.items():
        dtype = getattr(field, 'dataType', '')
        print(f"    - {field_id} : {field.name} : {dtype}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We demonstrate extraction for every record set discovered above, identified by their `@id`. This enables flexible future exploration.

In [ ]:
# Extract a DataFrame for each record set
dataframes = {}
# record_sets is guaranteed to be present from previous cell

for record_set_id in record_sets:
    records_iter = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_iter)
    dataframes[record_set_id] = df
    col_list = df.columns.tolist()
    print(f"Columns for record set {record_set_id}: {col_list[:10]}{'...' if len(col_list) > 10 else ''}")
    print(df.head(2), "\n")
# Select the first record set for demonstration
main_record_set_id = list(record_sets.keys())[0]
print(f"Main record set in use: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

All references to fields/columns are by their `@id`. Adjust the field ids below using the actual output from earlier cells if you want to explore other fields.

In [ ]:
# Select a numeric field for analysis. Example: "cr:coveragePercent" (please adapt to your dataset)
# List columns again for reference
main_df = dataframes[main_record_set_id]
print('Columns available:', main_df.columns.tolist())

# Try to identify a likely numeric field by inspecting column names
import re
possible_numeric_fields = [col for col in main_df.columns if re.search(r'int|float|count|percent|mw|abund|num|weight|[0-9]', col, re.I)]
print(f"Possible numeric fields: {possible_numeric_fields}")

# For demonstration, let's use the first matching field
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
else:
    numeric_field_id = main_df.select_dtypes('number').columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter: e.g. only keep rows where field > 10 (or nonzero, if all small numbers)
main_df_clean = main_df.copy()
main_df_clean[numeric_field_id] = pd.to_numeric(main_df_clean[numeric_field_id], errors='coerce')

threshold = 10  # Adjust according to actual data scale
filtered_df = main_df_clean[main_df_clean[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize numeric field
norm_field = f"{numeric_field_id}_normalized"
filtered_df[norm_field] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"First records for normalized {numeric_field_id}:")
print(filtered_df[[numeric_field_id, norm_field]].head())

# Try grouping by a text/categorical field if present
# Pick the first non-numeric field as group, e.g. "cr:accession"
group_fields = [col for col in main_df.columns if col != numeric_field_id and main_df[col].dtype == object]
if group_fields:
    group_field = group_fields[0]
    grouped = filtered_df.groupby(group_field, observed=True)[numeric_field_id].mean().reset_index()
    print(f"Grouped by {group_field} (mean {numeric_field_id}):")
    print(grouped.head(5))

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

We use matplotlib (and seaborn if available) for visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id} (> {threshold})")
plt.show()

# If categorical group field exists, plot violin or box plot
if 'group_field' in locals() and group_field in filtered_df:
    plt.figure(figsize=(12, 4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


## 6. Conclusion

In this notebook, we explored the FAIR² mass spectrometry dataset for human mast cell extracellular vesicles:

- Loaded Croissant metadata and reviewed the schema entities by `@id`.
- Showed extraction into DataFrames for each record set, with all field references by `@id` for robust/portable code.
- Performed EDA by filtering and normalizing numeric data, and grouping by a candidate category.
- Visualized numeric field distributions and compared groups (if present).

This workflow provides a robust, reproducible way to interact with FAIR data by referencing all core entities (record sets, fields/columns, etc.) by their unique identifiers from the schema, supporting data traceability and interpretability.